# AI Codebooks

In [30]:
# Import json
import json
with open('../schema/0.1.0/study.json') as f:
    schema = json.load(f)


# Setup endpoint and env for api key
endpoint = "https://genai-fa2026-resource-1.services.ai.azure.com/api/projects/Ethan_Kile_GenAI_Project"

# pip install python-dotenv
import os
from dotenv import load_dotenv
load_dotenv()  # loads variables from .env into environment

# Setup model

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="Kimi-K2.5",
    base_url=endpoint + "/openai/v1",
    api_key=os.getenv("API_KEY"),
    temperature=0,
    timeout=300,
    max_tokens=16000
)

system_prompt = f"""You are a synthetic data generator.
Return ONLY valid JSON, no markdown, no explanation.
Use 'Item.Type' as the field name, NOT 'Type'.
Match this schema exactly: {schema}

RULES:
- Return a JSON object, NOT a list at the top level
- 'Conditional.Type' must be one of: '1','2','3','4','5','6' or null — never a boolean
- Do not invent new field names

"""
query = "generate me data for a study looking to understand the longitudinal process of craving with negative affect over 10 days. Generate me data for 1 person."



In [19]:
# Run model
import json
from jsonschema import validate, ValidationError
from langchain_core.messages import SystemMessage, HumanMessage

max_retries = 3
messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=query)
]

for attempt in range(max_retries):
    response = llm.invoke(messages)
    try:
        raw = response.content.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        data = json.loads(raw)
        validate(instance=data, schema=schema)
        print("✓ Valid on attempt", attempt + 1)
        break  # success — exit the loop
    except (ValidationError, json.JSONDecodeError) as e:
        print(f"✗ Attempt {attempt + 1} failed: {e.message if hasattr(e, 'message') else str(e)}")
        if attempt == max_retries - 1:
            raise
        # Feed the error back so the model can self-correct
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": f"That was invalid JSON. Error: {e.message if hasattr(e, 'message') else str(e)}. Please fix and try again, returning only valid JSON."})

APITimeoutError: Request timed out.

In [33]:
import json

with open("study_output.json", "w") as f:
    json.dump(data, f, indent=2)

print("Saved to study_output.json")

Saved to study_output.json


In [17]:
import pandas as pd
itemRepository = pd.read_csv('itemRepository.csv')
print(df.head()) # Shows first 5 rows


                                 Article  \
0  Positive and Negative Affect Schedule   
1                                    NaN   
2                                    NaN   
3  Positive and Negative Affect Schedule   
4  Positive and Negative Affect Schedule   

                         What does it conceptualize?  \
0  "[The PANAS] was developed to be a brief easil...   
1                                                NaN   
2                                                NaN   
3                                                NaN   
4                                                NaN   

   Usual modeling characteristics Is it a statistically validated measure?  \
0                             NaN                                        1   
1                             NaN                                      NaN   
2                             NaN                                      NaN   
3                             NaN                                      NaN   
4       

In [31]:
system_prompt = f"""You are a synthetic study generator.
Return ONLY valid JSON, no markdown, no explanation.
Use 'Item.Type' as the field name, NOT 'Type'.
Match this schema exactly: {schema}

Additionally, you may reference this {itemRepository} item repository. It contains metadata about scales and items used by scientists. If one of the scales or items is applicable, generate based on the provided info. If there is not applicable items, generate new one's.

RULES:
- Return a JSON object, NOT a list at the top level
- 'Conditional.Type' must be one of: '1','2','3','4','5','6' or null — never a boolean
- Do not invent new field names

"""
query = "Generate me a study looking to understand the longitudinal process of people in recovery. Create 1 survey that is delivered 10 times daily. For this survey, Create questions that will cover a wide range of the possible daily processes that a recovery researcher may care about. Additionally, create baseline assessment survey with questions that are measured at 1 time point for each person. This baseline assessment should ask all relevant covariate information. Things like demographics, ses, drug use history, etc."


In [32]:
# Run model
import json
from jsonschema import validate, ValidationError
from langchain_core.messages import SystemMessage, HumanMessage

max_retries = 3
messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=query)
]

for attempt in range(max_retries):
    response = llm.invoke(messages)
    try:
        raw = response.content.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        data = json.loads(raw)
        validate(instance=data, schema=schema)
        print("✓ Valid on attempt", attempt + 1)
        break  # success — exit the loop
    except (ValidationError, json.JSONDecodeError) as e:
        print(f"✗ Attempt {attempt + 1} failed: {e.message if hasattr(e, 'message') else str(e)}")
        if attempt == max_retries - 1:
            raise
        # Feed the error back so the model can self-correct
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": f"That was invalid JSON. Error: {e.message if hasattr(e, 'message') else str(e)}. Please fix and try again, returning only valid JSON."})

✗ Attempt 1 failed: 'Item.Text' is a required property
✗ Attempt 2 failed: None is not of type 'string'
✓ Valid on attempt 3
